In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv(

"/Users/wuchaoqun/Desktop/Project1/raw data/customer_order_analysis.csv"

)


df.head()

,order_id,customer_unique_id,order_purchase_timestamp,product_id,price,freight_value,order_amount,payment_value,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,87285b34884572647811a353c7ac498a,29.99,8.72,38.71,18.12,sao paulo,SP
1,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,87285b34884572647811a353c7ac498a,29.99,8.72,38.71,2.00,sao paulo,SP
2,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,87285b34884572647811a353c7ac498a,29.99,8.72,38.71,18.59,sao paulo,SP
3,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,595fac2a385ac33a80bd5114aec74eb8,118.70,22.76,141.46,141.46,barreiras,BA
4,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,aa4383b373c6aca5d8797843e5594415,159.90,19.22,179.12,179.12,vianopolis,GO


In [4]:
df['order_purchase_timestamp']=pd.to_datetime(

df['order_purchase_timestamp']

)

In [5]:
analysis_date = (

df['order_purchase_timestamp'].max()

+
pd.Timedelta(days=1)

)


analysis_date

Timestamp('2018-08-30 15:00:37')

In [6]:
rfm = df.groupby(

'customer_unique_id'

).agg(

{

'order_purchase_timestamp':
lambda x:
(analysis_date-x.max()).days,


'order_id':
'count',


'payment_value':
'sum'

}

)

In [7]:
rfm.columns=[

'Recency',

'Frequency',

'Monetary'

]


rfm.head()

,Recency,Frequency,Monetary
customer_unique_id,,,
0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90
0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19
0000f46a3911fa3c0805444483337064,537,1,86.22
0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62
0004aac84e0df4da2b147fca70cf8255,288,1,196.89


In [8]:
rfm.describe()

,Recency,Frequency,Monetary
count,93358.000000,93358.000000,93358.000000
mean,237.941773,1.232224,211.831449
std,152.591453,0.819779,642.163458
min,1.000000,1.000000,0.000000
25%,114.000000,1.000000,63.760000
50%,219.000000,1.000000,112.950000
75%,346.000000,1.000000,201.740000
max,714.000000,75.000000,109312.640000


In [9]:
rfm['R_score']=pd.qcut(

rfm['Recency'],

5,

labels=[5,4,3,2,1]

)

In [10]:
rfm['F_score']=pd.qcut(

rfm['Frequency'].rank(method='first'),

5,

labels=[1,2,3,4,5]

)

In [11]:
rfm['M_score']=pd.qcut(

rfm['Monetary'],

5,

labels=[1,2,3,4,5]

)

In [12]:
rfm['RFM_score']= (

rfm['R_score'].astype(int)

+

rfm['F_score'].astype(int)

+

rfm['M_score'].astype(int)

)

In [13]:
def customer_segment(row):


    if row['RFM_score']>=12:

        return "VIP"


    elif row['RFM_score']>=9:

        return "Potential"


    elif row['Frequency']==1:

        return "New Customer"


    else:

        return "Lost Customer"



rfm['Segment']=rfm.apply(

customer_segment,

axis=1

)

In [14]:
rfm['Segment'].value_counts()

Segment
New Customer     40694
Potential        35307
VIP              16883
Lost Customer      474
Name: count, dtype: int64

In [15]:
rfm.to_csv(

"/Users/wuchaoqun/Desktop/Project1/raw data/rfm_customer_segment.csv"

)